<a href="https://colab.research.google.com/github/juliaduboisas/plnEp1/blob/main/roBERTaModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Inovando com XLM-RoBERTa

Por que usar o **RoBERTa** em vez do BERT tradicional (como o BERTimbau)?
1. **Treinamento Mais Robusto:** O RoBERTa otimizou os hiperparâmetros do BERT original e treinou com lotes (batches) maiores e muito mais dados.
2. **Máscaras Dinâmicas:** Durante o pré-treino, a forma como ele esconde palavras para tentar adivinhar muda constantemente, forçando o modelo a aprender representações melhores do texto.
3. **Multilíngue Forte:** O `xlm-roberta-base` foi treinado em mais de 100 idiomas (incluindo uma enorme base em Português), o que o torna excelente em capturar o sentido de frases e lidar bem com vocabulário fora do padrão (como gírias e jargões que podem aparecer no e-SIC).

Abaixo, vamos configurar a base do nosso classificador.

In [5]:
# Instalando as bibliotecas necessárias caso não estejam no ambiente
!pip install transformers datasets evaluate accelerate -q

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Vamos usar o XLM-RoBERTa Base, que é excelente para classificação em português
model_name = "xlm-roberta-base"

print(f"Carregando Tokenizador e Modelo: {model_name}...")

# 1. Carregar o Tokenizador
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Carregar o Modelo já com um "cabeçalho" de classificação para 3 classes
# Nossas classes são: c1, c234 e c5
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label={0: "c1", 1: "c234", 2: "c5"},
    label2id={"c1": 0, "c234": 1, "c5": 2}
)

print("Sucesso! Modelo preparado. O próximo passo seria tokenizar os dados de train.csv e passar para o Trainer.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
Carregando Tokenizador e Modelo: xlm-roberta-base...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Sucesso! Modelo preparado. O próximo passo seria tokenizar os dados de train.csv e passar para o Trainer.


### Passo 1: Obtenção dos Dados
Abaixo, clonamos o repositório contendo o arquivo `train.csv` com as respostas do e-SIC.

In [6]:
# Baixando todos os arquivos e dados do repositório do GitHub
!git clone https://github.com/juliaduboisas/plnEp1.git

Cloning into 'plnEp1'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 20 (delta 2), reused 15 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 7.39 MiB | 19.31 MiB/s, done.
Resolving deltas: 100% (2/2), done.


### Passo 2: Carregamento e Divisão do Dataset
Aqui lemos o CSV usando Pandas, mapeamos as classes (c1=0, c234=1, c5/c6=2) e dividimos os dados em treino (80%) e validação (20%).

In [7]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# 1. Carregar os dados
df = pd.read_csv('/content/plnEp1/train.csv')

# 2. Mapear rótulos para IDs (c1 -> 0, c234 -> 1, c5/c6 -> 2)
label_map = {"c1": 0, "c234": 1, "c5": 2, "c6": 2} # Considerando que c5 e c6 podem ser a mesma classe agrupada no dataset
df['label'] = df['clarity'].map(label_map)

# Remover possíveis nulos
df = df.dropna(subset=['resp_text', 'label'])
df['label'] = df['label'].astype(int)

# 3. Dividir em treino e validação (80/20)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# 4. Converter para o formato da biblioteca HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

print(f"Tamanho do treino: {len(train_dataset)}")
print(f"Tamanho da validação: {len(val_dataset)}")

Tamanho do treino: 16073
Tamanho da validação: 4019


### Passo 3: Otimização de Texto (512 Tokens)
Para evitar que respostas longas sejam cortadas, aumentamos o limite de leitura do modelo para o máximo permitido (512 tokens).

In [8]:
print("Aumentando o limite de tokens para 512...")

def tokenize_function_512(examples):
    return tokenizer(examples["resp_text"], padding="max_length", truncation=True, max_length=512)

# Re-tokenizando os dados com 512
tokenized_train_512 = train_dataset.map(tokenize_function_512, batched=True)
tokenized_val_512 = val_dataset.map(tokenize_function_512, batched=True)

# Limpando colunas sobressalentes
tokenized_train_512 = tokenized_train_512.remove_columns(['resp_text', 'clarity', '__index_level_0__'] if '__index_level_0__' in tokenized_train_512.column_names else ['resp_text', 'clarity'])
tokenized_val_512 = tokenized_val_512.remove_columns(['resp_text', 'clarity', '__index_level_0__'] if '__index_level_0__' in tokenized_val_512.column_names else ['resp_text', 'clarity'])

Aumentando o limite de tokens para 512...


Map:   0%|          | 0/16073 [00:00<?, ? examples/s]

Map:   0%|          | 0/4019 [00:00<?, ? examples/s]

---
### ⚠️ Baseline Antigo (128 Tokens - Teste Inicial)
*As próximas 3 células são do nosso primeiro teste (onde obtivemos um F1 mais baixo). Você pode mantê-las aqui como registro ou histórico, mas **não precisa rodá-las** para treinar o modelo otimizado.*

In [9]:
def tokenize_function(examples):
    # Trunca textos longos e faz padding para o tamanho máximo do lote
    return tokenizer(examples["resp_text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizando dataset de treino...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)

print("Tokenizando dataset de validação...")
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# Remover colunas desnecessárias para o modelo
tokenized_train = tokenized_train.remove_columns(['resp_text', 'clarity', '__index_level_0__'] if '__index_level_0__' in tokenized_train.column_names else ['resp_text', 'clarity'])
tokenized_val = tokenized_val.remove_columns(['resp_text', 'clarity', '__index_level_0__'] if '__index_level_0__' in tokenized_val.column_names else ['resp_text', 'clarity'])

Tokenizando dataset de treino...


Map:   0%|          | 0/16073 [00:00<?, ? examples/s]

Tokenizando dataset de validação...


Map:   0%|          | 0/4019 [00:00<?, ? examples/s]

In [10]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

# Métrica (Macro F1) importante para lidar com dados desbalanceados
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="macro")

# Argumentos de treinamento
training_args = TrainingArguments(
    output_dir="./xlm-roberta-results",
    eval_strategy="epoch", # Parâmetro corrigido para versões mais recentes da biblioteca
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3, # 3 épocas para um bom teste
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("Trainer configurado. Execute trainer.train() para iniciar o fine-tuning.")

Trainer configurado. Execute trainer.train() para iniciar o fine-tuning.


In [11]:
print("Treinamento baseline (128 tokens) ignorado para economizar tempo.")
# print("Iniciando o treinamento do XLM-RoBERTa...")
# trainer.train()
# print("Treinamento concluído!")

# Avaliando no conjunto de validação
# metrics = trainer.evaluate()
# print(f"\nMétricas na validação: {metrics}")

Treinamento baseline (128 tokens) ignorado para economizar tempo.


---
### Passo 4: Tratando o Desbalanceamento (Class Weights)
Como algumas classes aparecem muito mais que outras, calculamos pesos para 'punir' o modelo de forma mais severa quando ele erra as classes raras. Também configuramos um `Trainer` customizado para aplicar essa regra.

In [12]:
import torch
from torch import nn
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from transformers import Trainer # Importando aqui para garantir que esteja definido

# 1. Calcular pesos das classes para lidar com o desbalanceamento
labels_array = np.array(train_dataset['label'])
classes = np.unique(labels_array)
weights = compute_class_weight('balanced', classes=classes, y=labels_array)

# Mover pesos para a GPU (se disponível)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

print(f"Pesos calculados para as classes: {class_weights}")

# 2. Criar um Trainer customizado para injetar esses pesos na Loss Function
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Aplica os pesos das classes na CrossEntropyLoss
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

Pesos calculados para as classes: tensor([1.0551, 0.9773, 0.9718], device='cuda:0')


### Passo 5: Treinamento Otimizado Final
Aqui limpamos a memória da placa de vídeo e executamos o treinamento definitivo com 512 tokens e balanceamento de classes!

In [13]:
import gc

# Limpando a memória da GPU para evitar Out Of Memory (OOM) com segurança
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Recarregando modelo virgem...")
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label={0: "c1", 1: "c234", 2: "c5"},
    label2id={"c1": 0, "c234": 1, "c5": 2}
).to(device)

# Novos argumentos otimizados para 512 tokens
training_args_v2 = TrainingArguments(
    output_dir="./xlm-roberta-results-v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8, # Menor devido ao texto maior
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2, # Compensa o batch size menor
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(), # Acelera MUITO o treinamento na GPU
    report_to="none"
)

trainer_v2 = WeightedTrainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=tokenized_train_512,
    eval_dataset=tokenized_val_512,
    compute_metrics=compute_metrics,
)

print("O treinamento V2 intermediário foi comentado para pularmos direto para o Optuna!")
# trainer_v2.train()

# print("\nAvaliando o novo modelo...")
# metrics_v2 = trainer_v2.evaluate()
# print(f"Novas Métricas na validação: {metrics_v2}")

Recarregando modelo virgem...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


O treinamento V2 intermediário foi comentado para pularmos direto para o Optuna!


### Passo 6: Otimização de Hiperparâmetros (Grid/Random Search com Optuna)
Aqui usamos a biblioteca Optuna para encontrar a melhor combinação matemática para o nosso treinamento, maximizando o F1-Score.

In [14]:
!pip install optuna -q

In [15]:
import optuna

# Para o search funcionar, o Trainer precisa de uma função que gere um modelo NOVO a cada teste
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label={0: "c1", 1: "c234", 2: "c5"},
        label2id={"c1": 0, "c234": 1, "c5": 2}
    ).to(device)

# Redefinimos o trainer, agora passando 'model_init' ao invés de 'model'
trainer_optuna = WeightedTrainer(
    model=None, # O modelo será criado dinamicamente a cada tentativa
    model_init=model_init,
    args=training_args_v2,
    train_dataset=tokenized_train_512,
    eval_dataset=tokenized_val_512,
    compute_metrics=compute_metrics,
)

# Definimos o espaço de busca (Grid)
def my_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 5),
        "weight_decay": trial.suggest_float("weight_decay", 0.01, 0.1),
        # Mantendo batch size 8 para evitar erro de memória na GPU
    }

print("Iniciando a busca pelos melhores hiperparâmetros. Isso vai demorar algumas horas...")

best_run = trainer_optuna.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=my_hp_space,
    n_trials=5 # Número de combinações que ele vai testar. Aumente se tiver tempo (ex: 10)
)

print("\n========================================")
print("MELHORES PARÂMETROS ENCONTRADOS:")
print(best_run.hyperparameters)
print("========================================")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[I 2026-09-28 11:52:49,974] A new study created in memory with name: no-name-66c3edfc-d2dd-480

Iniciando a busca pelos melhores hiperparâmetros. Isso vai demorar algumas horas...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-09-28 14:32:45,834] Trial 4 finished with value: 0.16957328385899814 and parameters: {'learning_rate': 3.444750290711621e-05, 'num_train_epochs': 4, 'weight_decay': 0.020884761520613082}. Best is trial 0 with value: 0.4518448485497529.



MELHORES PARÂMETROS ENCONTRADOS:
{'learning_rate': 1.3372122850634388e-05, 'num_train_epochs': 3, 'weight_decay': 0.020324035023946932}


### Passo 7: Treinamento do Modelo Campeão (Definitivo)
Agora que o Optuna encontrou os melhores hiperparâmetros, vamos utilizá-los automaticamente para treinar a nossa versão final do modelo.

In [16]:
import gc
import torch

print("Limpando memória da GPU...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Fallback caso o Optuna não tenha terminado
if 'best_run' in globals():
    best_params = best_run.hyperparameters
    print(f"Parâmetros Vencedores do Optuna: {best_params}")
else:
    print("Busca do Optuna não encontrada. Usando parâmetros ótimos predefinidos...")
    best_params = {
        "learning_rate": 2e-5,
        "num_train_epochs": 4,
        "weight_decay": 0.01
    }

print("Carregando o modelo para o treinamento final...")
final_model = model_init()

# Usando os parâmetros exatos (ou os predefinidos)
final_training_args = TrainingArguments(
    output_dir="./xlm-roberta-champion",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=best_params["learning_rate"],
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=best_params["num_train_epochs"],
    weight_decay=best_params["weight_decay"],
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

final_trainer = WeightedTrainer(
    model=final_model,
    args=final_training_args,
    train_dataset=tokenized_train_512,
    eval_dataset=tokenized_val_512,
    compute_metrics=compute_metrics,
)

print("\nIniciando o treinamento do Modelo Campeão...")
final_trainer.train()

print("\nAvaliando o Modelo Campeão (Validação Final)...")
final_metrics = final_trainer.evaluate()
print(f"\n========================================")
print(f"MÉTRICAS FINAIS DO MODELO CAMPEÃO:")
print(f"Macro F1-Score: {final_metrics['eval_f1']:.4f}")
print(f"========================================")

Limpando memória da GPU...
Parâmetros Vencedores do Optuna: {'learning_rate': 1.3372122850634388e-05, 'num_train_epochs': 3, 'weight_decay': 0.020324035023946932}
Carregando o modelo para o treinamento final...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Iniciando o treinamento do Modelo Campeão...


Epoch,Training Loss,Validation Loss,F1
1,2.193697,1.086402,0.342616
2,2.131159,1.064999,0.413106
3,2.064456,1.059254,0.441569


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Avaliando o Modelo Campeão (Validação Final)...


Training Loss,Validation Loss,Epoch,F1
2.064456,1.059254,3,0.441569



MÉTRICAS FINAIS DO MODELO CAMPEÃO:
Macro F1-Score: 0.4416
